# 3)Hybrid method Primal-Dual Net + Total Variation for Deblur and Denoise operations
### Using Primal-Dual Net+TV to address noise and blur issues in images: network training and evaluation
_NOTE: This code was written to run on Google Colab. Any repetitions such as loading files at the beginning of section 6 and 7 are due to being able to re-run the network operation without running the entire code (including training)_

In [ ]:
from google.colab import drive
import os
from datasets import load_from_disk
from dataclasses import dataclass
from pathlib import Path
import sys
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau

import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import shutil
from typing import List, Tuple, Any

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
!pip install astra-toolbox




In [ ]:
### --- DIRECTORIES --- ###
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/Computational Imaging")
DATASET_DIR = BASE_DIR/"dataset"
RESULT_DIR = BASE_DIR/"hybrid_output"
LOSS_HISTORY_DIR = RESULT_DIR/"loss_history"
METRICS_DIR = RESULT_DIR/"metrics_data"
WEIGHTS_DIR = RESULT_DIR/"weights"
WEIGHTS_PATH = WEIGHTS_DIR/"PD_Net_and_TV.pth"
RECONSTRUCTION_DIR = RESULT_DIR/"reconstruction_examples"
LOCAL_DATASET = Path("/content/dataset_local")

IPPY_CONTAINER = BASE_DIR / "IPPy"

sys.path.append(str(BASE_DIR))


# Loop to fix same-named subfolder problem (IPPy)
for key in list(sys.modules.keys()):
    if "IPPy" in key or "utilities" in key:
        del sys.modules[key]

if str(IPPY_CONTAINER) not in sys.path:
    sys.path.insert(0, str(IPPY_CONTAINER))


from IPPy import utilities
from IPPy import operators2
import IPPy.unrolled2 as unrolled2
from IPPy.operators2 import Operator
from IPPy.utilities.metrics import PSNR, SSIM


### --- HYPERPARAMETERS --- ###
BATCH_SIZE               = 4
EPOCH_NUMBER             = 30
LEARNING_RATE            = 1e-4
PATIENCE_EARLY_STOPPING  = 7
NUM_ITERATIONS           = 7    # PD-Net iterations number
CNN_FEATURES             = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Loading data from the dataset
Pre-computed degraded inputs ensure a fair, standardized comparison across all models while optimizing computational efficiency. The dataset's extensive semantic variety prevents overfitting, forcing the network to learn generalized reconstruction features. Loading and personalization of dataset

In [ ]:
# Necessary to remove corrupt data due to data upload interruptions
if LOCAL_DATASET.exists() and not (LOCAL_DATASET / "dataset_info.json").exists():
    shutil.rmtree(str(LOCAL_DATASET))

# Dataset caching standard
if not LOCAL_DATASET.exists():
    shutil.copytree(str(DATASET_DIR), str(LOCAL_DATASET))

full_dataset = load_from_disk(str(LOCAL_DATASET))

train_data = full_dataset["train"]
test_data = full_dataset["test"]
validation_data = full_dataset["validation"]

print(f"Train size: {len(train_data)}")
print(f"Test size: {len(test_data)}")
print(f"Validation size: {len(validation_data)}")
#print(f"{train_data.features}")


# Creating a virtual expansion of dataset
class DegradedDataset(Dataset):
    def __init__(self, dataset_subset): # torch.utils.data.Dataset __init__ override
        self.data = dataset_subset # dataset_subset: base arrow dataset containing clean and noisy images.
        self.to_tensor = transforms.ToTensor()
        self.noise_cols = ["y_005", "y_010", "y_050", "y_100"]

    def __len__(self): #  torch.utils.data.Dataset __len__ override
        return len(self.data) * len(self.noise_cols) # Setting virtual dataset size 4 times bigger

    def __getitem__(self, idx):  # torch.utils.data.Dataset __getitem__ override
        img_idx   = idx // len(self.noise_cols) # Entire division -> value updated every 4 iterations, row index (source image index)
        noise_idx = idx % len(self.noise_cols) # Remainder operator to cycle through noise levels

        sample = self.data[img_idx]
        clean   = self.to_tensor(sample["x"].convert("RGB"))
        degraded = self.to_tensor(sample[self.noise_cols[noise_idx]].convert("RGB"))

        return  degraded, clean

    def getAll(self, idx):
        sample = self.data[idx]

        clean   = self.to_tensor(sample["x"].convert("RGB"))
        dg_005 = self.to_tensor(sample[self.noise_cols[0]].convert("RGB"))
        dg_010 = self.to_tensor(sample[self.noise_cols[1]].convert("RGB"))
        dg_050 = self.to_tensor(sample[self.noise_cols[2]].convert("RGB"))
        dg_100 = self.to_tensor(sample[self.noise_cols[3]].convert("RGB"))

        return (dg_005,dg_010,dg_050,dg_100),clean


train_dataset = DegradedDataset(train_data)
val_dataset = DegradedDataset(validation_data)
test_dataset = DegradedDataset(test_data)

# num_workers=2 delegates data loading and tensor conversion to 2 child processes.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,num_workers=2, drop_last=True) # Shuffle training data at the start of each epoch to ensure randomness and prevent overfitting
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)



Train size: 4160
Test size: 520
Validation size: 520


## 3. Sanity Check (Execution is optional)
extraction of a batch of images to verify the correct correspondency between ground truth images and degraded ones and to check the correctness of dataloader overall

In [ ]:
data_iter = iter(train_loader)
degraded_batch, clean_batch = next(data_iter)

print(f"Clean image tensor:   {clean_batch.shape}\n")
print(f"Degraded image tensor: {degraded_batch.shape}\n")

fig, axes = plt.subplots(4, 2, figsize=(10, 16))
for i in range(4):
    img_degraded = degraded_batch[i].permute(1, 2, 0).numpy()
    img_clean    = clean_batch[i].permute(1, 2, 0).numpy()

    axes[i, 0].imshow(img_clean)
    axes[i, 0].set_title(f"Index {i} - Ground Truth")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(img_degraded)
    axes[i, 1].set_title(f"Index {i} - Degraded Input")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


## Basic Function and Configuration

In [ ]:
@dataclass(frozen=True)
class Config:
  seed: int = 42
  dataset_name: str = "benjamin-paine/imagenet-1k-256x256"
  target_classes: List[int] = (0, 10, 100, 200) # default_factory with lambda to have each istance of Config it's own target_classes
  samples_per_class: int = 1300 #max

  drive_output_dir: Path = BASE_DIR / "dataset"

  # Forward operator parameters
  img_shape: Tuple[int, int, int] = (3, 256, 256) # img type, (C,H,W)
  blur_kernel_type: str = "gaussian"
  blur_kernel_size: int = 9
  blur_sigma: float = 2.0
  motion_angle: float = 45.0 # unused rn

  noise_levels: List[float] = (0.005, 0.01, 0.05, 0.1)
  device: str = device

def set_seed(seed: int = 42) -> None:
  """ Set seed for reproducibility """
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False
  torch.use_deterministic_algorithms(True, warn_only=True)

config = Config()
set_seed(config.seed)
blur = operators2.Blurring(
  img_shape=config.img_shape,
  kernel_type=config.blur_kernel_type,
  kernel_size=config.blur_kernel_size,
  kernel_variance=config.blur_sigma**2
)


## Model Factory

In [ ]:
### --- MODEL FACTORY --- ###
def build_hybrid_model():
    model = unrolled2.HybridLearnedPrimalDualTV(
        operator_A     = blur,
        img_shape      = (config.img_shape[1], config.img_shape[2]),
        num_iterations = NUM_ITERATIONS,
        primal_channels= 3,
        dual_channels  = 3,
        num_cnn_features = CNN_FEATURES,
    )

    # Xavier init per conv1, zero-init per conv2 (delta starts from zero → training stabile)
    def init_weights(m):
        if isinstance(m, nn.Conv2d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    model.apply(init_weights)

    for net in model.primal_nets:
        nn.init.zeros_(net.conv2.weight)
        nn.init.zeros_(net.conv2.bias)
    for net in model.dual_nets:
        nn.init.zeros_(net.conv2.weight)
        nn.init.zeros_(net.conv2.bias)

    return model

## Training Loop Definition

In [ ]:
def training_loop(model, w_path=WEIGHTS_PATH, num_epochs=EPOCH_NUMBER,
                  learning_rate=LEARNING_RATE, patience_early_stopping=PATIENCE_EARLY_STOPPING):

    model     = model.to(config.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    loss_fn   = nn.MSELoss()

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss  = float('inf')
    epochs_no_improve = 0

    for epoch in range(num_epochs):

        # --- Train ---
        model.train()
        train_loss = 0.0
        train_bar  = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)

        for degraded_batch, clean_batch in train_bar:
            degraded_batch = degraded_batch.to(config.device)
            clean_batch    = clean_batch.to(config.device)

            prediction = model(degraded_batch, x_init=degraded_batch.clone())
            loss       = loss_fn(prediction, clean_batch)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            train_bar.set_postfix({'loss': f'{loss.item():.6f}'})

        avg_train_loss = train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for degraded_batch, clean_batch in val_loader:
                degraded_batch = degraded_batch.to(config.device)
                clean_batch    = clean_batch.to(config.device)
                prediction     = model(degraded_batch, x_init=degraded_batch.clone())
                val_loss      += loss_fn(prediction, clean_batch).item()

        avg_val_loss = val_loss / len(val_loader)
        history['val_loss'].append(avg_val_loss)

        scheduler.step(avg_val_loss)

        # Salva i pesi migliori ed early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss     = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), w_path)
            try:
                history_path = LOSS_HISTORY_DIR / "history.json"
                with open(history_path, 'w') as f:
                    json.dump(history, f, indent=4)
            except Exception as e:
                pass
            tqdm.write(f"Epoch {epoch+1}/{num_epochs} -> Train: {avg_train_loss:.6f} | Val: {avg_val_loss:.6f} [BEST MODEL SAVED]")
        else:
            epochs_no_improve += 1
            tqdm.write(f"Epoch {epoch+1}/{num_epochs} -> Train: {avg_train_loss:.6f} | Val: {avg_val_loss:.6f} [No improvement: {epochs_no_improve} epochs]")

        if epochs_no_improve >= patience_early_stopping:
            tqdm.write(f"\nEarly Stopping activated. No improvements since {patience_early_stopping} epochs.")
            break

    # Ricarica i pesi migliori
    best_model = build_hybrid_model()
    best_model.load_state_dict(torch.load(w_path, map_location='cpu', weights_only=True))
    best_model = best_model.to(config.device).eval()

    # Salva history
    history_path = LOSS_HISTORY_DIR / "history.json"
    try:
        with open(history_path, 'w') as f:
            json.dump(history, f, indent=4)
        tqdm.write(f"[INFO] History saved in: {history_path}")
    except Exception as e:
        tqdm.write(f"[ERROR] Salvataggio history: {e}")

    return best_model, history


## Test Loop Execution

In [ ]:
torch.manual_seed(42)
hybrid_model = build_hybrid_model()
hybrid_model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device, weights_only=True))
hybrid_model, history = training_loop(hybrid_model)

## Model Evaluation

In [ ]:
if 'hybrid_model' not in locals() and 'hybrid_model' not in globals(): # Reload Model from drivel
    print("[INFO] 'hybrid_model' not found in RAM. Restoring from weights...")
    hybrid_model = build_hybrid_model()
    state_dict = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
    hybrid_model.load_state_dict(state_dict)
    hybrid_model = hybrid_model.to(device).eval()
    print("[SUCCESS] Pre-trained weights loaded safely.")
else:
    print("[INFO] 'hybrid_model' is already active in RAM. Ready for plotting.")
    hybrid_model.eval()



def evaluate_model(model, test_data, device=config.device):
    model.eval()
    to_tensor  = transforms.ToTensor()
    noise_cols = ["y_005", "y_010", "y_050", "y_100"]
    results    = {col: {"psnr": [], "ssim": []} for col in noise_cols}

    print("Valutazione in corso...")
    print(len(test_data))
    with torch.no_grad():
        for i in range(int(len(test_data)/4)):
            sample,cl    = test_data.getAll(i)
            clean_eval = cl.unsqueeze(0).to(device)

            for col in range(len(noise_cols)):
                degraded     = sample[col].unsqueeze(0).to(device)
                pred_eval    = model(degraded, x_init=degraded.clone()).clamp(0, 1)

                pred_cpu  = pred_eval.detach().cpu()
                clean_cpu = clean_eval.detach().cpu()

                results[noise_cols[col]]["psnr"].append(float(PSNR(pred_cpu, clean_cpu)))
                results[noise_cols[col]]["ssim"].append(float(SSIM(pred_cpu, clean_cpu)))

    import pandas as pd
    summary = {
        col: {
            "psnr_mean": np.mean(results[col]["psnr"]),
            "psnr_std":  np.std(results[col]["psnr"]),
            "ssim_mean": np.mean(results[col]["ssim"]),
            "ssim_std":  np.std(results[col]["ssim"]),
        }
        for col in noise_cols
    }
    return pd.DataFrame(summary).T, results


metrics_summary, raw_results = evaluate_model(hybrid_model, test_dataset)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
metrics_summary.to_csv(METRICS_DIR / "hybrid_metrics_per_noise_level.csv")
print(metrics_summary)

if 'history' not in locals() and 'history' not in globals(): # Reload loss history from dive
    if LOSS_HISTORY_DIR.exists():
        try:
            with open(LOSS_HISTORY_DIR/"history.json", 'r') as f:
              history = json.load(f)
            print(f"[SUCCESS] History uploaded from: {LOSS_HISTORY_DIR}")
        except Exception as e:
            print(f"[ERROR]: {e}")
    else:
      print("ERRORE: directory history non presente!")
else:
    print("History già presente")

### --- PLOTS --- ###
train_loss_hist = history['train_loss']
val_loss_hist   = history['val_loss']
train_psnr_hist = [-10 * np.log10(l) for l in train_loss_hist]
val_psnr_hist   = [-10 * np.log10(l) for l in val_loss_hist]
epochs_range    = range(1, len(train_loss_hist) + 1)

noise_levels_res = metrics_summary.index
psnr_means = metrics_summary['psnr_mean']
psnr_stds  = metrics_summary['psnr_std']
ssim_means = metrics_summary['ssim_mean']
ssim_stds  = metrics_summary['ssim_std']

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
ax1, ax2, ax3, ax4 = axes[0,0], axes[0,1], axes[1,0], axes[1,1]

ax1.plot(epochs_range, train_loss_hist, label='Train Loss', color='#1f77b4', linewidth=2)
ax1.plot(epochs_range, val_loss_hist,   label='Val Loss',   color='#ff7f0e', linewidth=2, linestyle='--')
ax1.set_title('Loss Curve (MSE)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epochs'); ax1.set_ylabel('MSE')
ax1.legend(); ax1.grid(True, linestyle=':', alpha=0.6)

ax2.plot(epochs_range, train_psnr_hist, label='Train PSNR', color='#2ca02c', linewidth=2)
ax2.plot(epochs_range, val_psnr_hist,   label='Val PSNR',   color='#d62728', linewidth=2, linestyle='--')
ax2.set_title('PSNR per Epoca', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epochs'); ax2.set_ylabel('PSNR (dB)')
ax2.legend(); ax2.grid(True, linestyle=':', alpha=0.6)

ax3.bar(noise_levels_res, psnr_means, yerr=psnr_stds, capsize=5,
        color='#2ca02c', edgecolor='black', alpha=0.8)
ax3.set_title('Test Set: PSNR per Noise Level', fontsize=14, fontweight='bold')
ax3.set_xlabel('Noise Column'); ax3.set_ylabel('PSNR (dB)')
ax3.grid(True, linestyle=':', alpha=0.6)

ax4.bar(noise_levels_res, ssim_means, yerr=ssim_stds, capsize=5,
        color='#9467bd', edgecolor='black', alpha=0.8)
ax4.set_title('Test Set: SSIM per Noise Level', fontsize=14, fontweight='bold')
ax4.set_xlabel('Noise Column'); ax4.set_ylabel('SSIM (0-1)')
ax4.set_ylim(0, 1.1); ax4.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig(METRICS_DIR / "hybrid_learning_curves.png", dpi=300)
plt.show()

## Image Result

In [ ]:
if 'hybrid_model' not in locals() and 'hybrid_model' not in globals(): # Reload Model from drive
    print("[INFO] 'hybrid_model' not found in RAM. Restoring from weights...")
    hybrid_model = build_hybrid_model()
    state_dict = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
    hybrid_model.load_state_dict(state_dict)
    hybrid_model = hybrid_model.to(device).eval()
    print("[SUCCESS] Pre-trained weights loaded safely.")
else:
    print("[INFO] 'hybrid_model' is already active in RAM. Ready for plotting.")
    hybrid_model.eval()


noise_cols = ["y_005", "y_010", "y_050", "y_100"]
chosen_idx = 76
sample,clean_tensor = test_dataset.getAll(chosen_idx)

to_tensor_vis  = transforms.ToTensor()
#clean_tensor   = to_tensor_vis(sample["x"].convert("RGB"))
img_clean      = clean_tensor.permute(1, 2, 0).numpy()

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
hybrid_model.eval()

for idx, col in enumerate(noise_cols):
    degraded_tensor = sample[idx]
    input_batch     = degraded_tensor.unsqueeze(0).to(config.device)

    with torch.no_grad():
        output_tensor = hybrid_model(input_batch, x_init=input_batch.clone()).squeeze(0).cpu().clamp(0, 1)

    axes[0, idx].imshow(degraded_tensor.permute(1, 2, 0).numpy())
    axes[0, idx].set_title(f"Input {col}", fontsize=14, fontweight='bold')
    axes[0, idx].axis("off")

    axes[1, idx].imshow(output_tensor.permute(1, 2, 0).numpy())
    axes[1, idx].set_title(f"Output {col}", fontsize=14, fontweight='bold', color='green')
    axes[1, idx].axis("off")

plt.tight_layout()
plt.savefig(RECONSTRUCTION_DIR / f"example_{chosen_idx}_grid.png", dpi=300)
plt.show()

plt.figure(figsize=(5, 5))
plt.imshow(img_clean)
plt.title("Ground Truth", fontsize=14, fontweight='bold')
plt.axis("off")
plt.savefig(RECONSTRUCTION_DIR / f"example_{chosen_idx}_groundtruth.png", dpi=300)
plt.show()

